In [0]:
WITH
REC AS
(
SELECT P.policy_key, P.start_date, L.line_of_insurance_key, L.coverage_key, L.as_lob_key, L.subline_key, L.insured_object_key
FROM prod_lakehouse.dev_john_armstrong_policy.V_POLICY_POLICY P
JOIN prod_lakehouse.dev_john_armstrong_policy.V_XREF_POLICY_LEVELS L 
  ON P.policy_key = L.policy_key AND P.start_date = L.start_date
WHERE UPPER(L.line_of_insurance_key) = 'PROPERTY'
)
, COV_DETAIL AS
(
SELECT REC.policy_key, REC.coverage_key
, REC.as_lob_key, REC.subline_key, C.coverage_code_key, T.transaction_amount_oc
, O.insured_object_number, O_PRNT.insured_object_number AS insured_object_number_PRNT
, REC.line_of_insurance_key, T.transaction_comment, T.measure_name, T.measure_detail_code
, OED_RGID.column_value AS RATINGGROUPID, OED_RGTY.column_value AS RATINGGROUPTYPE, RF_RKTY.risk_factor_value AS RISKTYPE, E_RKBI.exposure_value AS BITYPE
, CASE WHEN C.coverage_code_key IN (
'CollisionOptionA',
'CollisionOptionB',
'DischargeFromSewer',
'FireDepartmentServiceChargeCoverage',
'FloodRisk',
'MineSubsidence',
'MiscellaneousRealProperty',
'OTCOptionA',
'OTCOptionB',
'RatingGroupFloodRisk',
'Spoilage',
'TobaccoInSalesWarehouses',
'ValuablePapers'
) THEN 'N' ELSE 'Y' END AS USE_SUBCOVERAGECD
FROM REC REC
JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_TRANSACTION T 
  ON REC.policy_key = T.policy_key AND REC.start_date = T.start_date AND REC.coverage_key = T.coverage_key AND UPPER(T.measure_name) NOT LIKE '%BUREAU%'
JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_COVERAGE C 
  ON REC.coverage_key = C.coverage_key AND REC.start_date = C.start_date
LEFT JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_INSURED_OBJECT O 
  ON REC.policy_key = O.policy_key AND REC.start_date = O.start_date AND REC.insured_object_key = O.insured_object_key
LEFT JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_INSURED_OBJECT O_PRNT 
  ON REC.policy_key = O_PRNT.policy_key AND REC.start_date = O_PRNT.start_date AND O.insured_object_parent_key = O_PRNT.insured_object_key
LEFT JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_INSURED_OBJECT_EXTRA_DATA OED_RGID 
  ON REC.policy_key = OED_RGID.policy_key AND IF(REC.insured_object_key = '', 'X', REC.insured_object_key) = OED_RGID.insured_object_key AND REC.start_date = OED_RGID.start_date AND 'RATINGGROUPID' = UPPER(OED_RGID.column_name)
LEFT JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_INSURED_OBJECT_EXTRA_DATA OED_RGTY 
  ON REC.policy_key = OED_RGTY.policy_key AND OED_RGID.column_value = OED_RGTY.insured_object_key AND REC.start_date = OED_RGTY.start_date AND 'RATINGTYPE' = UPPER(OED_RGTY.column_name)
LEFT JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_RISK_FACTORS RF_RKTY 
  ON REC.policy_key = RF_RKTY.policy_key AND IF(REC.insured_object_key = '', 'X', REC.insured_object_key) = RF_RKTY.insured_object_key AND REC.start_date = RF_RKTY.start_date AND 'RISKTYPE' = UPPER(RF_RKTY.risk_factor_name)
LEFT JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_EXPOSURE E_RKBI 
  ON REC.policy_key = E_RKBI.policy_key AND REC.start_date = E_RKBI.start_date AND IF(REC.insured_object_key = '', 'X', REC.insured_object_key) = E_RKBI.insured_object_key AND 'BUSINESSINCOMEINPUTCOVERAGETYPE' = UPPER(E_RKBI.exposure_name)
WHERE T.transaction_amount_oc <> '0.000'
)
SELECT DISTINCT coverage_code_key AS COVERAGECD
, CASE WHEN USE_SUBCOVERAGECD = 'Y' THEN 
    RISKTYPE 
    || CASE WHEN BITYPE <> '' AND RISKTYPE <> '' THEN '-' || BITYPE ELSE '' END
    || CASE WHEN RATINGGROUPTYPE IS NOT NULL AND RISKTYPE <> '' THEN '-' || RATINGGROUPTYPE ELSE '' END
  ELSE '' END AS SUBCOVERAGECD
, as_lob_key, subline_key
, CASE WHEN (insured_object_number IS NULL OR insured_object_number = '') AND (insured_object_number_PRNT = '' OR insured_object_number_PRNT IS NULL) THEN '003' ELSE '004' END AS PAS_KEY_LEVEL
FROM COV_DETAIL
UNION ALL
SELECT DISTINCT T.transaction_comment AS COVERAGECD, ''
, 'NPR' AS as_lob_key, CASE WHEN UPPER(T.transaction_comment) IN ('KYMUNICIPALTAX') THEN 'TAX' ELSE 'SUR' END AS subline_key
, '001' AS PAS_KEY_LEVEL
FROM REC REC
JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_TRANSACTION T 
  ON REC.policy_key = T.policy_key AND REC.start_date = T.start_date AND UPPER(T.measure_name) NOT LIKE '%BUREAU%'
JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_TRANSACTION_ATTRIBUTES T_ATTR 
  ON T.policy_transaction_attributes_key = T_ATTR.policy_transaction_attributes_key AND 'TAXSTATE' = UPPER(T_ATTR.column_name)
WHERE T.transaction_comment <> ''
UNION ALL
SELECT DISTINCT T.measure_detail_code AS COVERAGECD, ''
, 'NPR', 'SUR'
, '003'
FROM REC REC
JOIN prod_lakehouse.dev_john_armstrong_policy.V_POLICY_TRANSACTION T 
  ON REC.policy_key = T.policy_key AND REC.start_date = T.start_date
WHERE UPPER(measure_name) IN ('FEECHANGE', 'FEE')
ORDER BY 1, 2, 4, 3